<a href="https://colab.research.google.com/github/macanh9602/AILearning/blob/main/cbow_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget https://storage.googleapis.com/laurencemoroney-blog.appspot.com/sarcasm.json

--2025-11-04 10:15:07--  https://storage.googleapis.com/laurencemoroney-blog.appspot.com/sarcasm.json
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.196.207, 173.194.216.207, 108.177.11.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.196.207|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2025-11-04 10:15:07 ERROR 404: Not Found.



In [ ]:
corpus = [
    'The sky is blue and beautiful.',
    'Love this blue and beautiful sky!',
    'The quick brown fox jumps over the lazy dog.',
    "A king's breakfast has sausages, ham, bacon, eggs, toast and beans",
    'I love green eggs, ham, sausages and bacon!',
    'The brown fox is quick and the blue dog is lazy!',
    'The sky is very blue and the sky is very beautiful today',
    'The dog is lazy but the brown fox is quick!'
]

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import text
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing import sequence # Thường dùng cho padding

# --- LỖI: 'corpus' CHƯA ĐƯỢC ĐỊNH NGHĨA ---
# Bạn cần định nghĩa 'corpus' trước khi chạy code bên dưới.
# corpus = [ 'câu 1 của bạn', 'câu 2 của bạn' ]
# ---------------------------------------------

# 1. Tạo và "fit" (huấn luyện) Tokenizer
tokenizer = text.Tokenizer()
tokenizer.fit_on_texts(corpus)

# 2. Lấy từ điển (bắt đầu từ index 1)
word2id = tokenizer.word_index

# 3. Build vocabulary of unique words
# Thêm token OOV (Out-of-Vocabulary) vào index 0
word2id['<OOV>'] = 0

# 4. Tạo từ điển ngược
id2word = {v: k for k, v in word2id.items()}

# 5. Chuyển đổi corpus thô thành 'wids' (danh sách các ID)
# (Lưu ý: 'text.text_to_word_sequence' là hàm mà 'fit_on_texts' dùng ngầm)
wids = [[word2id[w] for w in text.text_to_word_sequence(doc)] for doc in corpus]

# 6. Đặt các thông số
vocab_size = len(word2id)
embed_size = 100
window_size = 2 # context window size

# 7. In kết quả
print('Vocabulary Size:', vocab_size)
print('Vocabulary Sample:', list(word2id.items())[:10])

Vocabulary Size: 31
Vocabulary Sample: [('the', 1), ('is', 2), ('and', 3), ('sky', 4), ('blue', 5), ('beautiful', 6), ('quick', 7), ('brown', 8), ('fox', 9), ('lazy', 10)]


In [ ]:
vocab_size

31

In [ ]:
def generate_context_word_pairs(corpus, window_size, vocab_size):
    """
    Tạo ra các cặp (context, target) cho CBOW.
    Context là các từ xung quanh.
    Target là từ ở giữa.
    """
    total_window_size = window_size * 2

    for sentence in corpus:
        sentence_len = len(sentence)
        for i, target_word in enumerate(sentence):

            # 1. Tạo Context (Input - X)
            context_padded = []
            # Lấy context bên trái
            for k in range(window_size):
                index = i - (window_size - k)
                if index < 0:
                    context_padded.append(0) # Pad (đệm) bằng 0
                else:
                    context_padded.append(sentence[index])

            # Lấy context bên phải
            for k in range(window_size):
                index = i + k + 1
                if index >= sentence_len:
                    context_padded.append(0) # Pad (đệm) bằng 0
                else:
                    context_padded.append(sentence[index])

            # 2. Tạo Target (Output - Y)
            # Dùng to_categorical để tạo One-Hot vector (giống trong ảnh)
            target_one_hot = to_categorical([target_word], num_classes=vocab_size)

            # 3. Reshape X để vừa với input của model
            context_padded_np = np.array([context_padded]) # Shape (1, window_size*2)

            yield (context_padded_np, target_one_hot)

In [ ]:
# In 10 cặp (Context, Target) đầu tiên để kiểm tra
i = 0
for x, y in generate_context_word_pairs(corpus=wids, window_size=window_size, vocab_size=vocab_size):
    if 0 not in x[0]: # In ra các context không bị pad
        print(f'Context (X):', [id2word[w] for w in x[0]],
              '-> Target (Y):', id2word[np.argmax(y[0])])

    if i == 10:
        break
    i += 1

Context (X): ['the', 'sky', 'blue', 'and'] -> Target (Y): is
Context (X): ['sky', 'is', 'and', 'beautiful'] -> Target (Y): blue
Context (X): ['love', 'this', 'and', 'beautiful'] -> Target (Y): blue
Context (X): ['this', 'blue', 'beautiful', 'sky'] -> Target (Y): and


In [ ]:
# 1. build CBOW architecture
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Lambda, Dense
from tensorflow.keras import backend as K
from tensorflow.keras.utils import model_to_dot
from IPython.display import SVG

cbow = Sequential()
cbow.add(Embedding(input_dim=vocab_size, output_dim=embed_size, input_length=window_size*2))

# 2. Lớp Lambda: Lấy trung bình các vector context
# Đây là "trái tim" của CBOW (Continuous Bag of Words)
cbow.add(Lambda(lambda x: K.mean(x, axis=1), output_shape=(embed_size,)))

# 3. Lớp Output: Dự đoán từ ở giữa
cbow.add(Dense(vocab_size, activation='softmax'))

# 4. Compile
cbow.compile(loss='categorical_crossentropy', optimizer='rmsprop')

# 5. view model summary
print(cbow.summary())

# 6. visualize model structure
# Bạn có thể dùng cái này để vẽ sơ đồ model
# SVG(model_to_dot(cbow, show_shapes=True, show_layer_names=True).create(prog='dot', format='svg'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
print("Bắt đầu huấn luyện...")

# Chúng ta sẽ train 100 vòng (epochs)
for epoch in range(100):
    total_loss = 0

    # Tạo generator
    generator = generate_context_word_pairs(corpus=wids, window_size=window_size, vocab_size=vocab_size)

    # Train trên từng cặp (x, y)
    for x, y in generator:
        loss = cbow.train_on_batch(x, y)
        total_loss += loss

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/100, Loss: {total_loss}')

Bắt đầu huấn luyện...
Epoch 10/100, Loss: 225.71507263183594
Epoch 20/100, Loss: 199.98728942871094
Epoch 30/100, Loss: 179.14231872558594
Epoch 40/100, Loss: 161.66819763183594
Epoch 50/100, Loss: 146.86614990234375
Epoch 60/100, Loss: 134.21005249023438
Epoch 70/100, Loss: 123.2876968383789
Epoch 80/100, Loss: 113.78749084472656
Epoch 90/100, Loss: 105.48371887207031
Epoch 100/100, Loss: 98.20680236816406


In [ ]:
# Lấy ma trận trọng số (weights) từ lớp Embedding
# Đây chính là các vector embedding mà chúng ta cần
embedding_matrix = cbow.get_weights()[0]

print("Kích thước ma trận Embedding:", embedding_matrix.shape)

# In vector của từ 'fox' (ID=9)
print("Vector của từ 'fox':")
print(embedding_matrix[9])

Kích thước ma trận Embedding: (31, 100)
Vector của từ 'fox':
[-0.6873286   0.23260535  0.31751484  0.05959043  0.02267324 -0.51903015
 -0.01428225 -0.9018786   0.79762846 -0.23106894  0.1460801  -0.4410776
 -0.08913902 -0.07448988  0.20900849 -0.3566123  -0.23326162  0.4681644
  0.53350943  0.05590602  0.8102079  -0.5133526  -0.36971188  1.0085982
  0.36233208  0.288013   -0.06021611  0.22137715 -0.2400327  -1.7488273
 -0.15336114 -0.46737537 -0.76328987  0.3678778  -0.10937353  0.17621396
  0.17856818 -0.26731372 -0.4242709  -0.20723593 -0.08302692 -0.5430796
  0.49914894  0.24460484  0.8302753  -0.36527994  0.10940241 -0.6461362
  0.6409249  -0.17549367 -0.5297223   0.45275274  1.0882109  -0.7184611
  0.08697271 -1.1392425   0.3026342  -0.8312403   0.81965184 -0.36119425
 -0.01302649 -0.2578024   0.37848878 -0.5075304   0.59487784 -0.3859634
 -0.06879527  0.26328358  0.18717104 -0.6597018  -0.16995844 -0.747668
  0.5565622  -0.1775664   0.33764845  0.2551225   0.3386834   1.2059375
 

In [ ]:
print(cbow.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (1, 4, 100)            │         3,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (1, 100)               │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (1, 31)                │         3,131 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,464 (48.69 KB)

 Trainable params: 6,231 (24.34 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 6,233 (24.35 KB)

None
